In [2]:
import matplotlib.pyplot as plt
import sys
import os
import importlib
import pandas as pd
import xarray as xr

# Add the parent directory to the path to import src as a package
sys.path.insert(0, os.path.abspath('..'))
from src import dataloader

importlib.reload(dataloader)
from src import utils
from src import export

importlib.reload(export)
from src.export import write_dyad_to_uniwaw_imported

%matplotlib widget 
plot_flag = False

# Example of saving data for multiple dyads to NCDF while creating the folder structure

In [3]:
input_folder = "/Users/admin/Library/CloudStorage/GoogleDrive-j.zygierewicz@uw.edu.pl/.shortcut-targets-by-id/1N4ySQ5GO6UE8fY2jnRkRUjBFm4XHrBRv/SYNCC-IN/WP4          - Joint study/UniWAW Data collection/UNIWAW_RAW_DATA"
export_folder = "/Users/admin/Library/CloudStorage/GoogleDrive-j.zygierewicz@uw.edu.pl/.shortcut-targets-by-id/1N4ySQ5GO6UE8fY2jnRkRUjBFm4XHrBRv/SYNCC-IN/WP4          - Joint study/UniWAW Data collection/UNIWAW_EEG_exported"

In [4]:
# create a list of dyads to export
dyades_to_export = [
    'W_000', 'W_016', 'W_033', 'W_049', 'W_086',
    'W_001', 'W_017', 'W_034', 'W_053', 'W_087',
    'W_002', 'W_018', 'W_035', 'W_057', 'W_088',
    'W_003', 'W_019', 'W_036', 'W_064', 'W_090',
    'W_004', 'W_020', 'W_037', 'W_071', 'W_091',
    'W_005', 'W_021', 'W_038', 'W_072', 'W_092',
    'W_006', 'W_022', 'W_039', 'W_073', 'W_093',
    'W_007', 'W_024', 'W_040', 'W_074', 'W_094',
    'W_008', 'W_025', 'W_041', 'W_075', 'W_096',
    'W_009', 'W_026', 'W_042', 'W_076', 'W_097',
    'W_010', 'W_027', 'W_043', 'W_077', 'W_104',
    'W_011', 'W_028', 'W_044', 'W_078', 'W_112',
    'W_012', 'W_029', 'W_045', 'W_079',
    'W_013', 'W_030', 'W_046', 'W_081',
    'W_014', 'W_031', 'W_047', 'W_084',
    'W_015', 'W_032', 'W_048', 'W_085'
]
# load the metadata file to get the list of all dyads and their corresponding movie durations
metadata_file = os.path.join(input_folder, "meta_data.csv")
metadata_df = pd.read_csv(metadata_file, sep=';')
# filter the dyades_to_export list to include only the dyads in the metadata file
dyades_to_export = [dyad for dyad in dyades_to_export if dyad in metadata_df['ID'].values]
dyades_to_export = sorted(dyades_to_export)
print(f"Exporting {len(dyades_to_export)} dyads: {dyades_to_export}")

Exporting 75 dyads: ['W_000', 'W_001', 'W_002', 'W_003', 'W_004', 'W_005', 'W_006', 'W_007', 'W_008', 'W_009', 'W_010', 'W_011', 'W_012', 'W_013', 'W_014', 'W_015', 'W_016', 'W_017', 'W_018', 'W_019', 'W_020', 'W_021', 'W_022', 'W_024', 'W_025', 'W_026', 'W_027', 'W_028', 'W_029', 'W_030', 'W_031', 'W_032', 'W_033', 'W_034', 'W_035', 'W_036', 'W_037', 'W_038', 'W_039', 'W_040', 'W_041', 'W_042', 'W_043', 'W_044', 'W_045', 'W_046', 'W_047', 'W_048', 'W_049', 'W_053', 'W_057', 'W_064', 'W_071', 'W_072', 'W_073', 'W_074', 'W_075', 'W_076', 'W_077', 'W_078', 'W_079', 'W_081', 'W_084', 'W_085', 'W_086', 'W_087', 'W_088', 'W_090', 'W_091', 'W_092', 'W_093', 'W_094', 'W_096', 'W_097', 'W_104']


In [ ]:
failed_dyads = []
for dyad in dyades_to_export:
    try:
        print(f"Exporting {dyad}...")
        write_dyad_to_uniwaw_imported(
            [dyad],
            input_data_path=input_folder,
            export_path=export_folder,
            load_et=False,
            eeg_filter_type='fir',
            time_margin=10,
            verbose=True,
        )
        print(f"Done: {dyad}")
    except Exception as e:
        failed_dyads.append((dyad, str(e)))
        print(f"Failed: {dyad} -> {e}")

print(f"Finished. Success: {len(dyades_to_export) - len(failed_dyads)}, Failed: {len(failed_dyads)}")
if failed_dyads:
    print("Failed dyads:")
    for dyad, err in failed_dyads:
        print(f"  - {dyad}: {err}")

Info z notatek:
- W_006: plik eeg prawie pusty, wg warsaw ids nie zrobione 
- W_007: plik niby jest ale wg warsaw ids nie było eeg   
- W_008: nie było robione H10  
- W_011: brak H10, pliki od eeg niby są ale wg warsaw ids nie było eeg, w komentarzach "Zapisała się 1 rozmowa "   
- W_018: komenarz w warsaw ids: brak synchronizacji kamer z opaską;
- W_025: komentarz: Cz nie kontaktowała 
- W_033: wg warsaw ids nie było ani H10, ani eeg   
- W_037: list index out of range
- W_045: More than 2 talks detected, something is wrong. 
- W_046: eeg i h10 tylko mama   
- W_073: nie ma pliku stage_timings.txt   
- W_090: plik niby jest ale wg warsaw ids nie było eeg   
- W_092: oddzielne nagrania eeg z rozmowy   
- W_093: plik niby jest ale wg warsaw ids nie było eeg   
- W_096: ecg bardzo małe pliki, ibi puste, wg warsaw ids H10 tylko mama

# Example of reading from ncdf file to xarray

> Note: batch export now also writes RMSSD modality files under `RMSSD/<dyad_id>/<member>/...` when RMSSD is available in the loaded multimodal data.